# Analisis Sentimen Ulasan Wayground

Buka notebook ini di Google Colab dengan runtime CPU. Siapkan **wayground_colab_input.zip**, lalu pilih **Runtime → Run all** dan unggah ZIP saat diminta. Paket memuat skrip proyek, kamus, anotasi pakar, serta data penelitian.

Pilihan pada sel konfigurasi:
- `raw_arsip` (bawaan): jalankan pemrosesan scraper dari CSV mentah, preprocessing Sastrawi, pelabelan leksikon v2 dan integrasi anotasi pakar, serta split berbasis grup.
- `scraping_baru`: ambil ulasan Google Play melalui scraper proyek. Filter periode penelitian 1 Januari 2024–30 Juni 2026 tetap berlaku. Cakupan hasil dan kecocokan anotasi pakar dapat berubah.
- `snapshot_v4`: pakai dataset dan split yang dibekukan untuk melatih ulang konfigurasi v4 dan membandingkan metrik dengan laporan proyek.

Model mengikuti konfigurasi final proyek: TF-IDF kata 1–3 gram, SVM linear C=10, One-vs-One. Threshold 1,0 berasal dari seleksi development historis v4. Notebook memakai keputusan penelitian tersebut untuk seluruh mode, sehingga tujuh eksperimen pemilihan konfigurasi tidak perlu dijalankan ulang. Evaluasi v4 bersifat diagnostik karena holdout memiliki paparan historis dan sebagian besar label berasal dari leksikon. Versi Python dan pustaka pada runtime Colab dapat memengaruhi hasil training ulang.

In [ ]:
MODE_DATA = "raw_arsip" # @param ["raw_arsip", "scraping_baru", "snapshot_v4"]
UNDUH_HASIL = False # @param {type:"boolean"}

import os
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime
from google.colab import files

ROOT = Path("/content/wayground")
ROOT.mkdir(exist_ok=True)
arsip = Path("/content/wayground_colab_input.zip")
if not arsip.exists():
    unggahan = files.upload()
    if arsip.name not in unggahan:
        raise ValueError(f"Unggah {arsip.name}.")
    arsip.write_bytes(unggahan[arsip.name])
shutil.unpack_archive(str(arsip), str(ROOT))
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
RUN = ROOT / "runs" / datetime.now().strftime("colab_%Y%m%d_%H%M%S_%f")
RUN.mkdir(parents=True)
print(RUN)

## 1. Instalasi dependensi

In [ ]:
# Virtual environment menjaga dependensi proyek terpisah dari kernel Colab.
venv = Path("/content/wayground_venv")
subprocess.run([sys.executable, "-m", "venv", str(venv)], check=True)
PY = venv / "bin/python"
inti = ["numpy==1.26.2", "pandas==2.1.4", "scipy==1.11.4"]
if sys.version_info >= (3, 13):
    inti = ["numpy", "pandas", "scipy"]
subprocess.run([str(PY), "-m", "pip", "install", "-q", *inti,
                "scikit-learn==1.7.2", "joblib==1.4.2", "matplotlib",
                "Sastrawi==1.0.1", "tqdm==4.66.1", "google-play-scraper==1.2.6"], check=True)

def jalankan(*args):
    subprocess.run([str(PY), "-u", *map(str, args)], cwd=ROOT, check=True, env=dict(os.environ, PYTHONPATH=str(ROOT), PYTHONIOENCODING="utf-8"))

## 2. Scraping atau pembangunan ulang data mentah

In [ ]:
if MODE_DATA in {"raw_arsip", "scraping_baru"}:
    raw = RUN / "raw"
    raw.mkdir(exist_ok=True)
    opsi = []
    if MODE_DATA == "raw_arsip":
        shutil.copy2(ROOT / "data/01_semua_data.csv", raw)
        opsi = ["--rebuild-from-raw"]
    jalankan("pipeline/scraper.py", "--output-dir", raw, *opsi)
    sumber = raw / "02_data_akhir.csv"
elif MODE_DATA == "snapshot_v4":
    print("Menggunakan snapshot v4 dan split tersimpan.")
else:
    raise ValueError(f"Mode data tidak dikenal: {MODE_DATA}")

## 3. Preprocessing, pelabelan, dan split

Skrip proyek menerapkan filter periode penelitian, deduplikasi, normalisasi, stemming, pelabelan otomatis, dan penggabungan label pakar yang cocok. Snapshot menyimpan jejak audit serta split development dan holdout. Tahap derivasi menggunakan threshold v4 sambil mempertahankan ID pada setiap partisi.

In [ ]:
if MODE_DATA == "snapshot_v4":
    snapshot = ROOT / "data/pipeline_output_v4"
else:
    persiapan = RUN / "preparation"
    snapshot = RUN / "selected"
    jalankan("scripts/prepare_dataset_snapshot.py", "--source", sumber,
             "--output", persiapan)
    jalankan("scripts/derive_threshold_snapshot.py", "--source", persiapan,
             "--output", snapshot, "--experiment",
             ROOT / "output_v4/experiments/experiment_2_threshold_results.json")

import json
manifest = json.loads((snapshot / "dataset_manifest.json").read_text(encoding="utf-8-sig"))
print(json.dumps({k: manifest[k] for k in ("class_distribution", "split") if k in manifest},
                 indent=2, ensure_ascii=False))

## 4. Training dan evaluasi

Grouped cross-validation berjalan pada development. Ulasan dengan teks hasil preprocessing yang sama berada di grup yang sama. Pipeline memasang TF-IDF ulang dalam setiap fold. Setelah CV, model dilatih pada seluruh development dan dievaluasi pada holdout.

Kode memakai konstruktor model penelitian dan menyimpan artefak untuk penggunaan di notebook.

In [ ]:
training = r'''
import json
import platform
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

from pipeline.aspect_detection import detect_aspect_batch
from scripts.train_final_model import build_final_pipeline
from shared.preprocessing import preprocess_text
from shared.splitting import assert_no_overlap, build_groups
from shared.final_config import FINAL_MODEL_CONFIG

snapshot, output = map(Path, sys.argv[1:3])
mode = sys.argv[3]
labels = ["negatif", "netral", "positif"]
colors = ["#bd493d", "#a68b4d", "#247c6c"]
target = "sentiment_label_final"
split = snapshot / "stage5_split_v2"
dev = pd.read_csv(split / "development.csv", encoding="utf-8-sig")
hold = pd.read_csv(split / "holdout.csv", encoding="utf-8-sig")
assert_no_overlap(dev, hold)
for name, part in [("development", dev), ("holdout", hold)]:
    part[target] = part[target].astype(str).str.strip().str.casefold()
    part["processed_text"] = part.processed_text.fillna("").astype(str).str.strip()
    if set(part[target]) != set(labels) or part.processed_text.eq("").any():
        raise ValueError(f"Periksa cakupan tiga kelas dan teks {name}.")
texts = dev.processed_text
groups = build_groups(dev)
if dev.groupby(target).processed_text.nunique().min() < 5:
    raise ValueError("Development perlu sedikitnya lima grup teks per kelas untuk CV.")
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
model = build_final_pipeline()
print(f"Cross-validation: {len(dev)} ulasan development", flush=True)
scores = cross_val_score(model, texts, dev[target], groups=groups, cv=cv,
                         scoring="f1_macro", error_score="raise")
print("Training final dan evaluasi holdout", flush=True)
model.fit(texts, dev[target])
pred = model.predict(hold.processed_text.astype(str))
cm = confusion_matrix(hold[target], pred, labels=labels)
report = classification_report(hold[target], pred, labels=labels,
                               output_dict=True, zero_division=0)
evaluation = {
    "mode": mode, "model": FINAL_MODEL_CONFIG,
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "development_rows": len(dev), "holdout_rows": len(hold),
    "development_macro_f1_folds": scores.tolist(),
    "development_macro_f1_mean": float(scores.mean()),
    "development_macro_f1_sd": float(scores.std(ddof=1)),
    "holdout_accuracy": accuracy_score(hold[target], pred),
    "holdout_macro_f1": f1_score(hold[target], pred, labels=labels, average="macro"),
    "classification_report": report,
    "confusion_matrix_labels": labels, "confusion_matrix": cm.tolist(),
    "holdout_label_sources": hold.label_source.value_counts().to_dict(),
}
(output / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
hold.assign(prediksi=pred).to_csv(output / "prediksi_holdout.csv", index=False)
joblib.dump(model, output / "model_tfidf_svm.joblib")

# Ringkasan deskriptif prediksi kalimat yang mengandung kata kunci aspek.
reviews = pd.concat([dev, hold], ignore_index=True)
print("Mendeteksi aspek dan memproses kalimat", flush=True)
aspects = detect_aspect_batch(reviews)
aspects["processed_text"] = aspects.sentence.map(preprocess_text)
aspects = aspects.loc[aspects.processed_text.str.strip().ne("")].copy()
aspects["prediksi"] = pd.Series(dtype=str)
if not aspects.empty:
    aspects["prediksi"] = model.predict(aspects.processed_text)
aspects.to_csv(output / "prediksi_aspek.csv", index=False)
summary = pd.crosstab(aspects.aspect, aspects.prediksi).reindex(columns=labels, fill_value=0)
summary.to_csv(output / "ringkasan_aspek.csv")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def simpan(fig, nama):
    fig.tight_layout()
    fig.savefig(output / nama, dpi=150, bbox_inches="tight")
    plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set(title="Confusion matrix holdout", xlabel="Prediksi", ylabel="Label acuan")
simpan(fig, "confusion_matrix.png")
fig, ax = plt.subplots(figsize=(8, 4))
pd.DataFrame(report).T.loc[labels, ["precision", "recall", "f1-score"]].plot.bar(ax=ax)
ax.set(title="Metrik holdout per kelas", ylim=(0, 1), xlabel="Kelas", ylabel="Skor")
simpan(fig, "metrik_per_kelas.png")
fig, ax = plt.subplots(figsize=(8, 4))
distribusi = pd.DataFrame({"development": dev[target].value_counts(),
                           "holdout": hold[target].value_counts()}).reindex(labels)
distribusi.plot.bar(ax=ax)
ax.set(title="Distribusi label per partisi", ylabel="Jumlah ulasan", xlabel="Kelas")
simpan(fig, "distribusi_label.png")
if not summary.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    summary.plot.barh(stacked=True, color=colors, ax=ax)
    ax.set(title="Prediksi sentimen per aspek", xlabel="Jumlah pasangan kalimat–aspek")
    simpan(fig, "sentimen_aspek.png")
'''
hasil = RUN / "hasil"
hasil.mkdir(exist_ok=True)
script = RUN / "latih_evaluasi.py"
script.write_text(training, encoding="utf-8")
# Skrip dalam folder run mengimpor modul proyek dari root.
env = dict(os.environ, PYTHONPATH=str(ROOT))
subprocess.run([str(PY), "-u", str(script), str(snapshot), str(hasil), MODE_DATA],
               cwd=ROOT, env=env, check=True)

## 5. Hasil dan visualisasi

Grafik aspek merangkum prediksi pada kalimat yang terdeteksi oleh aturan kata kunci, termasuk kalimat dari development. Satuan hitungnya pasangan kalimat–aspek karena satu kalimat bisa mencakup beberapa aspek. Grafik ini bersifat deskriptif.

In [ ]:
from IPython.display import Image, display
import pandas as pd

evaluasi = json.loads((hasil / "evaluation.json").read_text(encoding="utf-8"))
for nama in ["development_macro_f1_mean", "holdout_accuracy", "holdout_macro_f1"]:
    print(f"{nama} = {evaluasi[nama]:.2%}")
print("Sumber label holdout:", evaluasi["holdout_label_sources"])
if MODE_DATA == "snapshot_v4":
    print("Referensi laporan v4 — macro F1 development 79,40%, "
          "akurasi holdout 84,27%, macro F1 holdout 79,02%.")
display(pd.DataFrame(evaluasi["classification_report"]).T)
for nama in ["distribusi_label.png", "confusion_matrix.png",
             "metrik_per_kelas.png", "sentimen_aspek.png"]:
    if (hasil / nama).exists():
        display(Image(filename=str(hasil / nama)))

## 6. Coba prediksi ulasan baru dan ekspor hasil

In [ ]:
ULASAN = "Aplikasinya seru dan membantu saya belajar." # @param {type:"string"}
inferensi = '''
import sys
import joblib
from shared.preprocessing import preprocess_text

teks = preprocess_text(sys.argv[2])
if teks.strip():
    model = joblib.load(sys.argv[1])
    print("Hasil preprocessing:", teks)
    print("Prediksi:", model.predict([teks])[0])
else:
    print("Masukkan ulasan yang menghasilkan token setelah preprocessing.")
'''
jalankan("-c", inferensi, hasil / "model_tfidf_svm.joblib", ULASAN)

zip_hasil = shutil.make_archive(str(RUN), "zip", root_dir=RUN)
print("Arsip hasil:", zip_hasil)
if UNDUH_HASIL:
    files.download(zip_hasil)